# Sindhi RAG Hallucination Benchmark: one notebook (Part 1 data, Part 2 baselines, Part 3A proposed method)
**Part 1** rebuilds the corpus from Sindhi Wikipedia and then CHECKS that it is identical to the released `Dataset/corpus.csv` (1,281 paragraphs, 231 documents). If it prints `REPRODUCED`, continue to Part 2. If it prints `NOT IDENTICAL`, do not use it: upload the released `Dataset/corpus.csv` in Part 2 instead. Upload `sindhi_rag_test_set.csv` (from `Dataset/`) when Part 2 asks.

---
## Part 1: Corpus, paragraphs, document-level dev/test split

In [1]:
!pip install -q datasets pandas

In [2]:
import os, random, re, json
import pandas as pd
from datasets import load_dataset

SEED = 42
random.seed(SEED)

# Saving inside Colab (no Google Drive needed). Download is done in the last cell.
OUT = '/content/sindhi_rag/Dataset'
os.makedirs(OUT, exist_ok=True)
print('Saving to', OUT)

Saving to /content/sindhi_rag/Dataset


### 1. Load Sindhi Wikipedia (existing dataset)
Source: `wikimedia/wikipedia`, config `20231101.sd`. Licence: CC BY-SA. **Verify on the Hugging Face page that this config exists and note the licence in your data statement.**

In [3]:
wiki = load_dataset("wikimedia/wikipedia", "20231101.sd", split="train")
print(wiki)
print(wiki[0]["title"])
print(wiki[0]["text"][:300])

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

20231101.sd/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 17.6MB            

20231101.sd/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16928 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 16928
})
ڪڪڙ
ڪڪڙ ھڪ پکي جو نالو آهي جنھن کي عربي ۾ دجاج، فارسي ۾ خروس، هندي ۾ ڪڪڙي، مرغي ۽ ماڪيان چوندا آهن.ھي مشهور ۽ سانتيڪو گهرو پکي آهي. ڪڪڙيون جلد جلد آنا لاهينديون آهن، جي ماڻهو کائيندا به آهن. نر جي مٿي تي ڳاڙهي رنگ جي چوٽي (هلڪي ڪلنگي) ۽ ماديءَ جي مٿي تي ننڍي ٿيندي آهي. ٻليءَ کان ڏاڍو ڊڄندو آهي. اونهاري 


### 2. Keep long, informative articles

In [4]:
N_DOCS = 250          # settings that produced the released corpus (Dataset/corpus.csv)
MIN_CHARS = 800

arts = [(i, a["title"], a["text"]) for i, a in enumerate(wiki) if len(a["text"]) >= MIN_CHARS]
print("Articles with >= %d chars: %d" % (MIN_CHARS, len(arts)))
random.seed(SEED)   # re-seed so re-running this cell gives the SAME corpus
random.shuffle(arts)
arts = arts[:N_DOCS]
print("Using", len(arts), "articles")

Articles with >= 800 chars: 5703
Using 250 articles


### 3. Split articles into paragraphs of roughly 60-250 words

In [5]:
def chunk(text, lo=60, hi=250):
    paras = [p.strip() for p in re.split(r"\n\s*\n|\n", text) if p.strip()]
    chunks, cur = [], []
    for p in paras:
        cur.append(p)
        if len(" ".join(cur).split()) >= lo:
            chunks.append(" ".join(cur)); cur = []
    if cur and len(" ".join(cur).split()) >= lo:
        chunks.append(" ".join(cur))
    return [c for c in chunks if lo <= len(c.split()) <= hi]

rows = []
for n, (orig_i, title, text) in enumerate(arts):
    doc_id = "d%03d" % n
    for j, c in enumerate(chunk(text)):
        rows.append({"doc_id": doc_id, "para_id": "%s_p%d" % (doc_id, j), "title": title, "text": c})
corpus = pd.DataFrame(rows)
print("Paragraphs:", len(corpus), "| Documents:", corpus.doc_id.nunique())
corpus.head()

Paragraphs: 1281 | Documents: 231


,doc_id,para_id,title,text
0,d000,d000_p0,آصف جوڻو,عشق مجاز کان قومي (سنڌ) جي عشق جو سفر آصف جوڻو...
1,d000,d000_p1,آصف جوڻو,الائي ڪھڙي قسم جون انجيڪشنون ڊرپ ۾ ملائي لڳايو...
2,d000,d000_p2,آصف جوڻو,کاڌي نه کائڻ ۽ دست ۽ الٽي جي ڪري اهڙي ڪمزوري ...
3,d000,d000_p3,آصف جوڻو,اڄ به بي جان لاش جيان بيڊ تي پيو هيس ۽ اکيون ب...
4,d000,d000_p4,آصف جوڻو,سندس لهجو ڪنهن شفيق ماء جيان محسوس ٿيندو هو سن...


### 4. Document-level dev/test split (prevents leakage)
30% of documents for dev (tune prompts, few-shot examples), 70% for test (used only at the end).

In [6]:
doc_ids = sorted(corpus.doc_id.unique())
random.Random(SEED).shuffle(doc_ids)
n_dev = int(0.3 * len(doc_ids))
dev_docs, test_docs = set(doc_ids[:n_dev]), set(doc_ids[n_dev:])
assert dev_docs.isdisjoint(test_docs)
corpus["split"] = corpus.doc_id.apply(lambda d: "dev" if d in dev_docs else "test")
print(corpus.split.value_counts())
print("Dev docs:", len(dev_docs), "| Test docs:", len(test_docs))

split
test    1021
dev      260
Name: count, dtype: int64
Dev docs: 69 | Test docs: 162


### 5. Statistics for your Data table

In [7]:
corpus["n_words"] = corpus.text.str.split().str.len()
stats = corpus.groupby("split").agg(documents=("doc_id","nunique"), paragraphs=("para_id","count"), avg_words=("n_words","mean")).round(1)
stats

,documents,paragraphs,avg_words
split,,,
dev,69,260,105.6
test,162,1021,100.9


### 6. Create the annotation sheet (what you fill in)
Test: 170 paragraphs x 2 rows = 340 rows (so you can reach 300+ after dropping bad ones). Dev: 60 paragraphs x 2 rows = 120 rows.

In [8]:
def make_rows(split, n_par):
    sub = corpus[corpus.split == split].sample(n=min(n_par, (corpus.split == split).sum()), random_state=SEED)
    out = []
    for _, r in sub.iterrows():
        for k in range(2):
            out.append({"split": split, "doc_id": r.doc_id, "para_id": r.para_id, "paragraph_text": r.text})
    return out

sheet = pd.DataFrame(make_rows("test", 170) + make_rows("dev", 60))
sheet.insert(0, "id", ["q%03d" % (i+1) for i in range(len(sheet))])
for c in ["question_arabic","question_roman","answerable","gold_answer","evidence","q_type","topic","notes"]:
    sheet[c] = ""
print(sheet.shape)
sheet.head(3)

(460, 13)


,id,split,doc_id,para_id,paragraph_text,question_arabic,question_roman,answerable,gold_answer,evidence,q_type,topic,notes
0,q001,test,d060,d060_p2,عام روايتن جي مطابق پاڻ اسرائيلي نبي ھئا ۽ حضر...,,,,,,,,
1,q002,test,d060,d060_p2,عام روايتن جي مطابق پاڻ اسرائيلي نبي ھئا ۽ حضر...,,,,,,,,
2,q003,test,d087,d087_p5,آبھوا ڪوپن ڪلائميٽ جي درجابندي مطابق دادو جي م...,,,,,,,,


### 7. Save everything

In [9]:
corpus.to_csv(f"{OUT}/corpus.csv", index=False, encoding="utf-8-sig")
sheet.to_csv(f"{OUT}/annotation_sheet.csv", index=False, encoding="utf-8-sig")
stats.to_csv(f"{OUT}/corpus_stats.csv", encoding="utf-8-sig")
json.dump({"seed": SEED, "dev_docs": sorted(dev_docs), "test_docs": sorted(test_docs)}, open(f"{OUT}/split.json","w"))
print("Saved: corpus.csv, annotation_sheet.csv, corpus_stats.csv, split.json")

Saved: corpus.csv, annotation_sheet.csv, corpus_stats.csv, split.json


### 7a. Check that Part 1 rebuilt exactly the released corpus

In [10]:
# ---- Reproducibility check: is the regenerated corpus IDENTICAL to the released one? ----
import hashlib
_sha = lambda x: hashlib.sha1(x.encode("utf-8")).hexdigest()
EXPECT = {"rows": 1281, "docs": 231, "text_sha1": "d2a0e8560f548ae50ab5ec529e459c987ee2bc9c", "test_docs_sha1": "2c3806b90944aac81abbcb54a7fb31076ed7cebf"}
got = {"rows": len(corpus), "docs": corpus.doc_id.nunique(),
        "text_sha1": _sha("\n".join(sorted(corpus.text))), "test_docs_sha1": _sha(",".join(sorted(test_docs)))}
print("expected:", EXPECT["rows"], "paragraphs,", EXPECT["docs"], "documents")
print("got     :", got["rows"], "paragraphs,", got["docs"], "documents")
if got == EXPECT:
    corpus[["doc_id", "para_id", "title", "text", "split"]].to_csv("/content/corpus.csv", index=False, encoding="utf-8-sig")
    print("REPRODUCED: corpus and dev/test split are IDENTICAL to the released Dataset. /content/corpus.csv is ready for Part 2.")
else:
    print("NOT IDENTICAL to the released Dataset. Do NOT use this corpus (the 303 questions were written for the released one).")
    print("Upload the released Dataset/corpus.csv (1,281 rows) in Part 2 instead.")

expected: 1281 paragraphs, 231 documents
got     : 1281 paragraphs, 231 documents
REPRODUCED: corpus and dev/test split are IDENTICAL to the released Dataset. /content/corpus.csv is ready for Part 2.


### 8. Download your files (IMPORTANT: Colab deletes files when the session ends)

In [11]:
import shutil
shutil.make_archive('/content/sindhi_rag_part1', 'zip', '/content/sindhi_rag')
from google.colab import files
files.download('/content/sindhi_rag_part1.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Part 2: Retrieval baselines (BM25 and dense), Arabic-script vs Roman Sindhi queries
Upload `corpus.csv` and `sindhi_rag_test_set.csv` when asked (they are in the `Dataset/` folder). Metrics: Recall@k and MRR on the answerable questions.

In [12]:
!pip install -q rank_bm25 sentence-transformers

In [13]:
import os, re, time, json
import numpy as np, pandas as pd
from google.colab import files

for f in ["corpus.csv", "sindhi_rag_test_set.csv"]:
    if not os.path.exists(f):
        print("Please upload", f)
        files.upload()
corpus = pd.read_csv("corpus.csv", encoding="utf-8-sig").fillna("")
test = pd.read_csv("sindhi_rag_test_set.csv", encoding="utf-8-sig").fillna("")


import hashlib
_sha = lambda t: hashlib.sha1(t.encode("utf-8")).hexdigest()
_md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
same_corpus = len(corpus) == 1281 and _sha("\n".join(sorted(corpus.text))) == "d2a0e8560f548ae50ab5ec529e459c987ee2bc9c"
print("corpus.csv ->", "identical to the released corpus" if same_corpus else "DIFFERENT from the released corpus (use Dataset/corpus.csv)")
print("sindhi_rag_test_set.csv ->", "identical to the released file" if _md5("sindhi_rag_test_set.csv") == "0e9d9df7e510f1bd334f102ed58af972" else "DIFFERENT from the released file (use Dataset/sindhi_rag_test_set.csv)")

# ---- Safety check: every test question's paragraph must exist in the retrieval corpus ----
corpus_texts = set(corpus.text)
found = test.paragraph_text.isin(corpus_texts).mean()
print(f"Test paragraphs found (exact text) in corpus.csv: {found:.1%}")
if found < 1.0:
    print("WARNING: corpus.csv does not match the test set (maybe it came from a different run of Part 1).")
    print("The missing paragraphs are added to the corpus below so the results stay valid.")

# Build ONE retrieval corpus with fresh unique ids; gold paragraph is matched by TEXT, not by old ids
corpus = (pd.concat([corpus[["text"]], test[["paragraph_text"]].rename(columns={"paragraph_text": "text"})])
          .drop_duplicates("text").reset_index(drop=True))
corpus["para_id"] = ["P%04d" % i for i in range(len(corpus))]
text2id = dict(zip(corpus.text, corpus.para_id))
test["para_id"] = test.paragraph_text.map(text2id)

ans = test[test.answerable == "yes"].reset_index(drop=True)
ids = corpus.para_id.tolist()
assert test.para_id.notna().all()
print("paragraphs in index:", len(corpus), "| test questions:", len(test), "| answerable:", len(ans))

Please upload sindhi_rag_test_set.csv


Saving sindhi_rag_test_set.csv to sindhi_rag_test_set.csv
corpus.csv -> identical to the released corpus
sindhi_rag_test_set.csv -> identical to the released file
Test paragraphs found (exact text) in corpus.csv: 100.0%
paragraphs in index: 1278 | test questions: 303 | answerable: 245


### Script normalisation (Perso-Arabic Sindhi) and tokenizer

In [14]:
MAP = {"ھ":"ه","ہ":"ه","ة":"ه","ی":"ي","ى":"ي","ک":"ڪ","ك":"ڪ"}
def norm(t):
    for a, b in MAP.items():
        t = t.replace(a, b)
    return re.sub(r"[\u064B-\u065F\u0670\u200c\u200d]", "", t)   # diacritics, ZWNJ, ZWJ
def tok(t):
    return re.findall(r"\w+", norm(t).lower())

### Metrics

In [15]:
KS = (1, 3, 5, 10)
def score_ranking(rank_lists, gold_ids):
    """rank_lists: list of ranked para_id lists; gold_ids: gold para_id per question."""
    rec = {k: 0 for k in KS}; rr = 0.0
    for ranked, gold in zip(rank_lists, gold_ids):
        if gold in ranked:
            rr += 1 / (ranked.index(gold) + 1)
        for k in KS:
            if gold in ranked[:k]:
                rec[k] += 1
    n = len(gold_ids)
    out = {f"R@{k}": round(rec[k] / n, 3) for k in KS}
    out["MRR"] = round(rr / n, 3)
    return out

### Baseline 1 (classical): BM25

In [16]:
from rank_bm25 import BM25Okapi
t0 = time.time()
bm25 = BM25Okapi([tok(x) for x in corpus.text])
bm25_index_time = time.time() - t0

def bm25_rank(queries, topk=10):
    out = []
    for q in queries:
        s = bm25.get_scores(tok(q))
        out.append([ids[i] for i in np.argsort(-s)[:topk]])
    return out

results, latency = {}, {}
for col in ["question_arabic", "question_roman"]:
    t0 = time.time()
    ranks = bm25_rank(ans[col].tolist())
    latency[("BM25", col)] = (time.time() - t0) / len(ans)
    results[("BM25", col)] = score_ranking(ranks, ans.para_id.tolist())
    print("BM25", col, results[("BM25", col)])

BM25 question_arabic {'R@1': 0.841, 'R@3': 0.963, 'R@5': 0.963, 'R@10': 0.971, 'MRR': 0.896}
BM25 question_roman {'R@1': 0.082, 'R@3': 0.127, 'R@5': 0.143, 'R@10': 0.155, 'MRR': 0.107}


### Baseline 2 (neural): multilingual-e5 dense retrieval
This downloads `intfloat/multilingual-e5-base` (about 1 GB). Use a GPU runtime (Runtime, Change runtime type, T4 GPU).

In [17]:
import torch
from sentence_transformers import SentenceTransformer
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
enc = SentenceTransformer("intfloat/multilingual-e5-base", device=device)

t0 = time.time()
P = enc.encode(["passage: " + norm(t) for t in corpus.text], batch_size=32,
               normalize_embeddings=True, show_progress_bar=True)
dense_index_time = time.time() - t0
print("indexing seconds:", round(dense_index_time, 1))

def dense_rank(queries, topk=10):
    Q = enc.encode(["query: " + norm(q) for q in queries], batch_size=32, normalize_embeddings=True)
    S = Q @ P.T
    return [[ids[i] for i in np.argsort(-row)[:topk]] for row in S]

for col in ["question_arabic", "question_roman"]:
    t0 = time.time()
    ranks = dense_rank(ans[col].tolist())
    latency[("e5-base", col)] = (time.time() - t0) / len(ans)
    results[("e5-base", col)] = score_ranking(ranks, ans.para_id.tolist())
    print("e5-base", col, results[("e5-base", col)])

device: cuda


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

indexing seconds: 14.7
e5-base question_arabic {'R@1': 0.767, 'R@3': 0.906, 'R@5': 0.927, 'R@10': 0.947, 'MRR': 0.838}
e5-base question_roman {'R@1': 0.127, 'R@3': 0.229, 'R@5': 0.269, 'R@10': 0.322, 'MRR': 0.185}


### Results table (copy into the paper)

In [18]:
rows = []
for (sysname, col), m in results.items():
    rows.append({"system": sysname, "query": "Arabic script" if col == "question_arabic" else "Roman Sindhi",
                 **m, "latency_s_per_query": round(latency[(sysname, col)], 4)})
res_df = pd.DataFrame(rows)
res_df

,system,query,R@1,R@3,R@5,R@10,MRR,latency_s_per_query
0,BM25,Arabic script,0.841,0.963,0.963,0.971,0.896,0.0091
1,BM25,Roman Sindhi,0.082,0.127,0.143,0.155,0.107,0.0152
2,e5-base,Arabic script,0.767,0.906,0.927,0.947,0.838,0.0014
3,e5-base,Roman Sindhi,0.127,0.229,0.269,0.322,0.185,0.0016


In [19]:
res_df.to_csv("retrieval_results.csv", index=False)
# gap caused by script (for RQ2)
for s in ["BM25", "e5-base"]:
    a = results[(s, "question_arabic")]["R@5"]; r = results[(s, "question_roman")]["R@5"]
    print(s, "Recall@5 drop from Arabic to Roman:", round(a - r, 3))

BM25 Recall@5 drop from Arabic to Roman: 0.82
e5-base Recall@5 drop from Arabic to Roman: 0.658


### Breakdown by topic (fairness analysis)

In [20]:
def by_topic(rank_fn, col):
    out = {}
    for tp, g in ans.groupby("topic"):
        ranks = rank_fn(g[col].tolist())
        out[tp] = {"n": len(g), **score_ranking(ranks, g.para_id.tolist())}
    return pd.DataFrame(out).T
by_topic(bm25_rank, "question_arabic")

,n,R@1,R@3,R@5,R@10,MRR
geography,55.0,0.836,0.964,0.964,0.964,0.897
health,5.0,1.000,1.000,1.000,1.000,1.000
history_culture,156.0,0.827,0.962,0.962,0.974,0.885
science,29.0,0.897,0.966,0.966,0.966,0.931


In [21]:
# Save ranked lists for the answer-generation stage (Part 3)
saved = {}
for name, fn in [("bm25", bm25_rank), ("e5", dense_rank)]:
    for col in ["question_arabic", "question_roman"]:
        saved[f"{name}|{col}"] = dict(zip(test.id, fn(test[col].tolist(), topk=10)))   # all 303 questions
json.dump(saved, open("retrieval_rankings.json", "w"))
files.download("retrieval_results.csv"); files.download("retrieval_rankings.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Part 3A: Proposed retrieval method (script-agnostic consonant skeleton) + ablation, robustness, significance
Run this in the SAME session right after Part 2 (it reuses `corpus`, `ans`, `bm25_rank`, `dense_rank`, `results`).

**Idea.** Perso-Arabic Sindhi does not write short vowels, while Roman Sindhi does and spells them inconsistently. We map both scripts to the same consonant skeleton (e.g. `ڪيترو` and `ketro` both become `ktr`) and run BM25 over skeleton tokens.

In [22]:
import random
AR = {"ب":"b","ٻ":"b","ڀ":"B","پ":"p","ت":"t","ٽ":"t","ٿ":"T","ٺ":"t","ث":"s","ج":"j","ڄ":"j","ڃ":"n","چ":"C","ڇ":"C",
      "ح":"h","خ":"K","د":"d","ڌ":"D","ڏ":"d","ڊ":"d","ڍ":"D","ذ":"z","ر":"r","ڙ":"r","ز":"z","ژ":"z","س":"s","ش":"S","ص":"s",
      "ض":"z","ط":"t","ظ":"z","غ":"G","ف":"f","ڦ":"f","ق":"k","ڪ":"k","ک":"k","گ":"g","ڳ":"g","ڱ":"n","ل":"l","م":"m",
      "ن":"n","ڻ":"n","ھ":"h","ه":"h","ہ":"h","ة":"h"}
DIGR = [("kh","K"),("gh","G"),("ch","C"),("sh","S"),("th","T"),("dh","D"),("bh","B"),("ph","f"),("jh","j"),("ck","k")]
LAT = {"b":"b","c":"k","d":"d","f":"f","g":"g","h":"h","j":"j","k":"k","l":"l","m":"m","n":"n","p":"p","q":"k",
       "r":"r","s":"s","t":"t","v":"w","w":"w","z":"z","x":"ks"}
def _strip_h(s):
    return s[:-1] if len(s) > 1 and s.endswith("h") else s
def skel_ar(w):
    out = []
    for i, ch in enumerate(w):
        if ch == "و" and i == 0: out.append("w")
        elif ch in AR: out.append(AR[ch])
        elif ch.isdigit(): out.append(ch)
        elif ch.isascii() and ch.isalpha(): out.append(LAT.get(ch.lower(), ""))
    return _strip_h("".join(out))
def skel_lat(w):
    w = w.lower()
    for a, b in DIGR: w = w.replace(a, b)
    out = []
    for ch in w:
        if ch in "KGCSTDBf" or ch.isdigit(): out.append(ch)
        elif ch in LAT: out.append(LAT[ch])
    return _strip_h("".join(out))
def skeleton_tokens(text):
    toks = []
    for w in re.findall(r"\w+", norm(text)):
        s = skel_ar(w) if re.search(r"[\u0600-\u06FF\u0750-\u077F]", w) else skel_lat(w)
        s = re.sub(r"(.)\1+", r"\1", s)
        if s: toks.append(s)
    return toks

print(skeleton_tokens("ڪيترو ڪھڙي آهي دادو"), skeleton_tokens("ketro kahri aahe Dadu"))

['ktr', 'khr', 'h', 'd'] ['ktr', 'khr', 'h', 'd']


### Proposed system: BM25 over consonant skeletons

In [23]:
from rank_bm25 import BM25Okapi
bm25_sk = BM25Okapi([skeleton_tokens(x) for x in corpus.text])
def skel_rank(queries, topk=10):
    return [[ids[i] for i in np.argsort(-bm25_sk.get_scores(skeleton_tokens(q)))[:topk]] for q in queries]

gold = ans.para_id.tolist()
for col in ["question_arabic", "question_roman"]:
    t0 = time.time()
    ranks = skel_rank(ans[col].tolist())
    latency[("Skeleton-BM25 (ours)", col)] = (time.time() - t0) / len(ans)
    results[("Skeleton-BM25 (ours)", col)] = score_ranking(ranks, gold)
    print("Skeleton-BM25", col, results[("Skeleton-BM25 (ours)", col)])

Skeleton-BM25 question_arabic {'R@1': 0.824, 'R@3': 0.931, 'R@5': 0.959, 'R@10': 0.967, 'MRR': 0.88}
Skeleton-BM25 question_roman {'R@1': 0.702, 'R@3': 0.869, 'R@5': 0.902, 'R@10': 0.931, 'MRR': 0.79}


### Ablation: remove each component
Components: (1) skeleton normalisation, (2) fusion with dense retrieval (reciprocal rank fusion).

In [24]:
def rrf(lists, k=60, topk=10):
    sc = {}
    for L in lists:
        for r, p in enumerate(L):
            sc[p] = sc.get(p, 0) + 1 / (k + r + 1)
    return [p for p, _ in sorted(sc.items(), key=lambda x: -x[1])][:topk]

variants = {}
for col in ["question_arabic", "question_roman"]:
    q = ans[col].tolist()
    r_plain, r_dense, r_skel = bm25_rank(q), dense_rank(q), skel_rank(q)
    variants[("Full: skeleton + dense (RRF)", col)] = [rrf([a, b]) for a, b in zip(r_skel, r_dense)]
    variants[("- fusion (skeleton only)", col)] = r_skel
    variants[("- skeleton (plain BM25 + dense RRF)", col)] = [rrf([a, b]) for a, b in zip(r_plain, r_dense)]
    variants[("- skeleton - fusion (plain BM25)", col)] = r_plain
    variants[("dense only (e5)", col)] = r_dense
abl = [{"variant": v, "query": "Arabic" if c == "question_arabic" else "Roman", **score_ranking(r, gold)}
       for (v, c), r in variants.items()]
abl_df = pd.DataFrame(abl)
abl_df

,variant,query,R@1,R@3,R@5,R@10,MRR
0,Full: skeleton + dense (RRF),Arabic,0.816,0.955,0.980,0.992,0.888
1,- fusion (skeleton only),Arabic,0.824,0.931,0.959,0.967,0.880
2,- skeleton (plain BM25 + dense RRF),Arabic,0.833,0.955,0.980,0.988,0.893
3,- skeleton - fusion (plain BM25),Arabic,0.841,0.963,0.963,0.971,0.896
4,dense only (e5),Arabic,0.767,0.906,0.927,0.947,0.838
5,Full: skeleton + dense (RRF),Roman,0.608,0.820,0.882,0.922,0.716
6,- fusion (skeleton only),Roman,0.702,0.869,0.902,0.931,0.790
7,- skeleton (plain BM25 + dense RRF),Roman,0.090,0.204,0.253,0.327,0.163
8,- skeleton - fusion (plain BM25),Roman,0.082,0.127,0.143,0.155,0.107
9,dense only (e5),Roman,0.127,0.229,0.269,0.322,0.185


### Significance test: paired bootstrap on Recall@5 (Roman queries)
Proposed system vs the best baseline (plain BM25 or e5, whichever is better on Roman queries).

In [25]:
def hit_at(ranks, k=5):
    return np.array([int(g in r[:k]) for r, g in zip(ranks, gold)])

def paired_bootstrap(a, b, n_boot=10000, seed=0):
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(a), len(a))
        diffs.append(a[idx].mean() - b[idx].mean())
    diffs = np.array(diffs)
    p = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return a.mean() - b.mean(), np.percentile(diffs, [2.5, 97.5]), min(p, 1.0)

col = "question_roman"
base = {"BM25": hit_at(bm25_rank(ans[col].tolist())), "e5-base": hit_at(dense_rank(ans[col].tolist()))}
best_name = max(base, key=lambda k: base[k].mean())
ours = hit_at(skel_rank(ans[col].tolist()))
d, ci, p = paired_bootstrap(ours, base[best_name])
print(f"Best baseline on Roman queries: {best_name} (R@5 = {base[best_name].mean():.3f})")
print(f"Skeleton-BM25 (ours): R@5 = {ours.mean():.3f} | difference = {d:+.3f}, 95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}], p = {p:.4f}")

Best baseline on Roman queries: e5-base (R@5 = 0.269)
Skeleton-BM25 (ours): R@5 = 0.902 | difference = +0.633, 95% CI [+0.567, +0.698], p = 0.0000


### Robustness: spelling noise on Roman queries (3 seeds)
Noise: for each word (with probability p) swap a vowel, drop a vowel, or double a letter.

In [26]:
V = "aeiou"
def noisy(q, p, rng):
    out = []
    for w in q.split():
        if rng.random() < p and len(w) > 2:
            op = rng.choice(["swap", "drop", "dbl"]); i = rng.randrange(len(w))
            if op == "swap" and w[i].lower() in V: w = w[:i] + rng.choice(V) + w[i+1:]
            elif op == "drop" and w[i].lower() in V: w = w[:i] + w[i+1:]
            elif op == "dbl": w = w[:i] + w[i] + w[i:]
        out.append(w)
    return " ".join(out)

rows = []
for p in [0.0, 0.3, 0.6]:
    per = {"BM25": [], "e5-base": [], "Skeleton-BM25 (ours)": []}
    for seed in [0, 1, 2]:
        rng = random.Random(seed)
        qs = [noisy(q, p, rng) for q in ans["question_roman"]]
        per["BM25"].append(hit_at(bm25_rank(qs)).mean())
        per["e5-base"].append(hit_at(dense_rank(qs)).mean())
        per["Skeleton-BM25 (ours)"].append(hit_at(skel_rank(qs)).mean())
    for s, v in per.items():
        rows.append({"noise_prob": p, "system": s, "R@5_mean": round(np.mean(v), 3), "R@5_std": round(np.std(v), 3)})
noise_df = pd.DataFrame(rows)
noise_df

,noise_prob,system,R@5_mean,R@5_std
0,0.0,BM25,0.143,0.000
1,0.0,e5-base,0.269,0.000
2,0.0,Skeleton-BM25 (ours),0.902,0.000
3,0.3,BM25,0.117,0.005
4,0.3,e5-base,0.224,0.010
5,0.3,Skeleton-BM25 (ours),0.899,0.004
6,0.6,BM25,0.105,0.008
7,0.6,e5-base,0.197,0.005
8,0.6,Skeleton-BM25 (ours),0.899,0.004


### Independent Roman test (written by you, not by AI)
Fill `question_roman_own` in `roman_own_template.csv` (60 rows) in your own natural Roman Sindhi, upload it here, and run this cell. This tests whether the method works on spelling that was not produced by the same author as the method's rules.

In [27]:
own_file = "roman_own_template.csv"
if not os.path.exists(own_file):
    print("Upload roman_own_template.csv (filled)")
    files.upload()
if os.path.exists(own_file):
    own = pd.read_csv(own_file, encoding="utf-8-sig").fillna("")
    own = own[own.question_roman_own.str.strip() != ""]
    sub = ans.merge(own[["id", "question_roman_own"]], on="id")
    print("answerable questions with your own Roman version:", len(sub))
    if len(sub) >= 20:
        g = sub.para_id.tolist(); q = sub.question_roman_own.tolist()
        for name, fn in [("BM25", bm25_rank), ("e5-base", dense_rank), ("Skeleton-BM25 (ours)", skel_rank)]:
            print(name, score_ranking(fn(q), g))
    else:
        print("Fill at least 20 rows to evaluate.")

Upload roman_own_template.csv (filled)


Saving roman_own_template.csv to roman_own_template.csv
answerable questions with your own Roman version: 52
BM25 {'R@1': 0.096, 'R@3': 0.135, 'R@5': 0.154, 'R@10': 0.173, 'MRR': 0.12}
e5-base {'R@1': 0.154, 'R@3': 0.269, 'R@5': 0.308, 'R@10': 0.365, 'MRR': 0.219}
Skeleton-BM25 (ours) {'R@1': 0.596, 'R@3': 0.827, 'R@5': 0.865, 'R@10': 0.885, 'MRR': 0.718}


In [28]:
all_res = pd.DataFrame([{"system": s, "query": "Arabic script" if c == "question_arabic" else "Roman Sindhi",
                          **m, "latency_s_per_query": round(latency[(s, c)], 4)} for (s, c), m in results.items()])
all_res.to_csv("retrieval_results_all.csv", index=False)
abl_df.to_csv("ablation_retrieval.csv", index=False)
noise_df.to_csv("robustness_noise.csv", index=False)
for f in ["retrieval_results_all.csv", "ablation_retrieval.csv", "robustness_noise.csv"]:
    files.download(f)
all_res

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,system,query,R@1,R@3,R@5,R@10,MRR,latency_s_per_query
0,BM25,Arabic script,0.841,0.963,0.963,0.971,0.896,0.0091
1,BM25,Roman Sindhi,0.082,0.127,0.143,0.155,0.107,0.0152
2,e5-base,Arabic script,0.767,0.906,0.927,0.947,0.838,0.0014
3,e5-base,Roman Sindhi,0.127,0.229,0.269,0.322,0.185,0.0016
4,Skeleton-BM25 (ours),Arabic script,0.824,0.931,0.959,0.967,0.880,0.0030
5,Skeleton-BM25 (ours),Roman Sindhi,0.702,0.869,0.902,0.931,0.790,0.0031


# Part 3B: Answer generation with an open LLM, hallucination measurement, human labels
Run in the SAME session after Part 2 and Part 3A (needs `corpus`, `test`, `text2id`, `bm25_rank`, `skel_rank`).
GPU required (Runtime, Change runtime type, T4 GPU).

**What is measured**
- Answerable questions: accuracy (gold answer contained in the answer, or token-F1 >= 0.5) and token-F1.
- Unanswerable questions: abstention accuracy (the model should say `NOT_FOUND`); `1 - abstention` is the hallucination rate on unanswerable questions.
- Unsupported-answer rate (automatic proxy for unfaithfulness): share of non-abstained answers whose tokens are mostly not found in the retrieved context. This proxy is validated against human labels at the end.

**Systems**: A = BM25 + Arabic query (upper bound), B = BM25 + Roman query (baseline), C = Skeleton-BM25 + Roman query (ours), D = C + faithfulness-check prompting (ours, full).

In [29]:
!pip install -q transformers accelerate scikit-learn

In [30]:
import random, re, math
from collections import Counter
if not os.path.exists("dev_set.csv"):
    print("Upload dev_set.csv (from Dataset/)")
    files.upload()
dev = pd.read_csv("dev_set.csv", encoding="utf-8-sig").fillna("").astype(str)
dev["pid"] = dev.paragraph_text.map(text2id)
assert dev.pid.notna().all(), "dev paragraphs must be in corpus.csv"
pid2text = dict(zip(corpus.para_id, corpus.text))
print("dev questions:", len(dev), "| few-shot pool:", int(dev.shot.astype(int).sum()))

# ---- experiment settings ----
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"
K_CTX    = 2          # paragraphs given to the LLM
N_SHOTS  = 2          # few-shot examples (1 answerable + 1 unanswerable)
SEEDS    = [0, 1, 2]  # seed controls which few-shot examples are drawn
SUBSET_N = 150        # number of test questions per run (None = all 303). Same subset for every system.

def pick_subset(n):
    if n is None: return test.id.tolist()
    rng = np.random.default_rng(0); out = []
    for a, g in test.groupby("answerable"):
        out += list(rng.choice(g.id.values, round(n * len(g) / len(test)), replace=False))
    return sorted(out)
EVAL_IDS = pick_subset(SUBSET_N)
eval_df = test[test.id.isin(EVAL_IDS)].reset_index(drop=True)
print("evaluation questions:", len(eval_df), eval_df.answerable.value_counts().to_dict())

Upload dev_set.csv (from Dataset/)


Saving dev_set.csv to dev_set.csv
dev questions: 28 | few-shot pool: 8
evaluation questions: 150 {'yes': 121, 'no': 29}


### Retrieval for every test question (top-K contexts)

In [31]:
retrieved = {}
for rname, fn in [("BM25", bm25_rank), ("Skeleton", skel_rank)]:
    for col in ["question_arabic", "question_roman"]:
        retrieved[(rname, col)] = dict(zip(test.id, fn(test[col].tolist(), topk=K_CTX)))
print("retrieval ready:", list(retrieved.keys()))

retrieval ready: [('BM25', 'question_arabic'), ('BM25', 'question_roman'), ('Skeleton', 'question_arabic'), ('Skeleton', 'question_roman')]


### Load the LLM (fp16 on the T4 GPU)

In [32]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
assert torch.cuda.is_available(), "Turn on the T4 GPU: Runtime, Change runtime type"
tokz = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16, device_map="auto")
llm.eval()
print("loaded", LLM_NAME, "| GPU memory (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

loaded Qwen/Qwen2.5-3B-Instruct | GPU memory (GB): 7.3


### Prompts, generation, faithfulness check

In [33]:
SYSTEM = ("You answer questions about Sindhi (Perso-Arabic script) documents. Use ONLY the given context. "
          "Answer in Sindhi with a short phrase or number copied from the context (at most 10 words). "
          "If the context does not contain the answer, reply exactly: NOT_FOUND")
def fmt_ctx(pids): return "\n\n".join(f"[{i+1}] {pid2text[p]}" for i, p in enumerate(pids))
def user_msg(q, pids): return f"Context:\n{fmt_ctx(pids)}\n\nQuestion: {q}\nAnswer:"
def build_messages(q, pids, shots):
    m = [{"role": "system", "content": SYSTEM}]
    for s in shots:
        m.append({"role": "user", "content": user_msg(s["question_arabic"], [s["pid"]])})
        m.append({"role": "assistant", "content": s["gold_answer"] if s["answerable"] == "yes" else "NOT_FOUND"})
    m.append({"role": "user", "content": user_msg(q, pids)})
    return m

@torch.no_grad()
def generate(messages, max_new_tokens=40):
    text = tokz.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokz(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokz.eos_token_id)
    return tokz.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip(), int(enc.input_ids.shape[1])

VERIFY_SYSTEM = "You are a strict fact checker. Reply with exactly one word: YES or NO."
def verify(q, pids, answer):
    msg = [{"role": "system", "content": VERIFY_SYSTEM},
           {"role": "user", "content": f"Context:\n{fmt_ctx(pids)}\n\nQuestion: {q}\nProposed answer: {answer}\n\n"
                                       "Is the proposed answer fully supported by the context? Reply YES or NO."}]
    out, _ = generate(msg, max_new_tokens=3)
    return out.strip().upper().startswith("YES")

shot_pool = dev[dev.shot.astype(int) == 1].to_dict("records")
def draw_shots(seed):
    rng = random.Random(seed)
    a = [s for s in shot_pool if s["answerable"] == "yes"]; u = [s for s in shot_pool if s["answerable"] == "no"]
    shots = [rng.choice(a), rng.choice(u)][:N_SHOTS]
    rng.shuffle(shots)
    return shots
def is_abstain(s): return "NOT_FOUND" in s.upper().replace(" ", "_")

### Metrics (answer normalisation, accuracy, F1, abstention, unsupported-answer proxy)

In [34]:
def nrm(s): return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", norm(str(s)).lower())).strip()
def toks(s): return nrm(s).split()
def tok_f1(pred, gold):
    p, g = toks(pred), toks(gold)
    c = sum((Counter(p) & Counter(g)).values())
    if not p or not g or c == 0: return 0.0
    pr, rc = c / len(p), c / len(g)
    return 2 * pr * rc / (pr + rc)
def is_correct(pred, gold): return nrm(gold) in nrm(pred) or tok_f1(pred, gold) >= 0.5
def support(pred, ctx_texts):
    pt = toks(pred); cs = set(toks(" ".join(ctx_texts)))
    return float(np.mean([t in cs for t in pt])) if pt else 0.0

def per_question(records):
    rows = []
    for r in records:
        t = test[test.id == r["id"]].iloc[0]
        ab = r["abstained"]; ctx = [pid2text[p] for p in r["retrieved"]]
        row = {"id": r["id"], "answerable": t.answerable, "topic": t.topic, "abstained": ab,
               "support": support(r["answer"], ctx) if not ab else np.nan}
        if t.answerable == "yes":
            row["correct"] = (not ab) and is_correct(r["answer"], t.gold_answer)
            row["f1"] = 0.0 if ab else tok_f1(r["answer"], t.gold_answer)
            row["gold_in_context"] = any(nrm(t.gold_answer) in nrm(c) for c in ctx)
        else:
            row["correct"] = ab          # correct behaviour = abstain
        row["unsupported"] = (not ab) and row["support"] < 0.5
        rows.append(row)
    return pd.DataFrame(rows)

def summarize(pq):
    a, u = pq[pq.answerable == "yes"], pq[pq.answerable == "no"]
    ans_rows = pq[~pq.abstained]
    return {"Acc_answerable": a.correct.mean(), "F1_answerable": a.f1.mean(),
            "Abstain_acc_unans": u.correct.mean(), "Halluc_unans": 1 - u.correct.mean(),
            "Unsupported_rate": ans_rows.unsupported.mean() if len(ans_rows) else np.nan,
            "Abstain_rate_answerable": a.abstained.mean()}

### Runner (saves every answer to disk, so a Colab disconnect does not lose work)

In [35]:
os.makedirs("gen_outputs", exist_ok=True)
RUNS = {}
def run_config(cfg, retr, col, mode, seed=0, verify_answers=False):
    key = f"{cfg}|{mode}|seed{seed}"
    path = f"gen_outputs/{re.sub(r'[^A-Za-z0-9_|.-]', '_', key)}.jsonl"
    done = {}
    if os.path.exists(path):
        for l in open(path, encoding="utf-8"): r = json.loads(l); done[r["id"]] = r
    shots = draw_shots(seed) if mode == "few-shot" else []
    t_start = time.time(); n_new = 0
    with open(path, "a", encoding="utf-8") as fh:
        for _, t in eval_df.iterrows():
            if t.id in done: continue
            pids = retrieved[(retr, col)][t.id]
            t0 = time.time()
            a, ntok = generate(build_messages(t[col], pids, shots))
            ab = is_abstain(a)
            if verify_answers and not ab:
                if not verify(t[col], pids, a): a, ab = "NOT_FOUND", True
            rec = {"id": t.id, "answer": a, "abstained": ab, "retrieved": pids, "prompt_tokens": ntok,
                   "secs": round(time.time() - t0, 3)}
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n"); done[t.id] = rec; n_new += 1
    RUNS[key] = [done[i] for i in eval_df.id]
    gpu_sec = sum(r["secs"] for r in RUNS[key])
    print(f"{key}: {len(RUNS[key])} answers ({n_new} new) | total GPU seconds in file: {gpu_sec:.0f}")
    return RUNS[key]

### Step 1: check the prompt on the DEV set (never on test) and measure speed

In [36]:
# zero-shot, gold paragraph as context: checks that the prompt/format works before the big runs
rows = []; t0 = time.time()
for _, d in dev.iterrows():
    a, _ = generate(build_messages(d.question_arabic, [d.pid], []))
    ab = is_abstain(a)
    ok = ab if d.answerable == "no" else ((not ab) and is_correct(a, d.gold_answer))
    rows.append({"q": d.question_arabic[:40], "answerable": d.answerable, "pred": a[:50], "ok": ok})
dev_out = pd.DataFrame(rows)
print("dev accuracy (gold context):", round(dev_out.ok.mean(), 3), "| sec/question:", round((time.time() - t0) / len(dev), 2))
dev_out.head(10)

dev accuracy (gold context): 0.571 | sec/question: 0.65


,q,answerable,pred,ok
0,جارج ايليٽ جو اصل نالو ڇا ھو؟,yes,ايونس هو,False
1,جارج ايليٽ ڪھڙي تاريخ تي ڄائي؟,yes,22 نومبر 1819,True
2,جارج ايليٽ جي وفات ڪھڙي سال ۾ ٿي؟,no,NOT_FOUND,True
3,زندگي براڙ ڪھڙي تاريخ تي ڄائو؟,yes,8 سيپٽمبر 1973,True
4,زندگي براڙ جي شادي ڪھڙي تاريخ تي ٿي؟,yes,18 فبروري 2020 تي,True
5,گارا گرائيف جو جنم ڪھڙي شھر ۾ ٿيو؟,yes,فيبروري 5,False
6,گرائيف ڪيترا موسيقي جا حصا لکيا؟,yes,110 موسيقي جا حصا لکيا,True
7,گارا گرائيف ڪھڙي يونيورسٽيءَ ۾ پڙھيو؟,no,1918 باڪو۾,False
8,علاؤ الدين ڪھڙي سال ۾ وفات ڪئي؟,yes,8 ــ 1307ع,True
9,علاؤ الدين جي وفات کان پوءِ تخت تي ويھار,yes,8 ــ 1307ع,False


In [37]:
# projected time for the whole plan (3 systems x (1 zero-shot + 3 few-shot seeds) + 3 verify runs)
sec_q = (time.time() - t0) / len(dev)
runs_total = 3 * (1 + len(SEEDS)) + len(SEEDS)
print(f"~{sec_q*1.6:.1f} s/question with retrieval contexts -> about {runs_total * len(eval_df) * sec_q * 1.6 / 3600:.1f} hours for {runs_total} runs of {len(eval_df)} questions")
print("If this is too long: lower SUBSET_N (e.g. 100) in the settings cell, or run only some systems below.")

~1.6 s/question with retrieval contexts -> about 1.0 hours for 15 runs of 150 questions
If this is too long: lower SUBSET_N (e.g. 100) in the settings cell, or run only some systems below.


### Step 2: the main runs
Systems A, B, C (zero-shot and few-shot with 3 seeds), then D (C + faithfulness check, few-shot). You can run one cell at a time; finished runs are reused if you re-run.

In [38]:
SYSTEMS = {"A: BM25 + Arabic query":        ("BM25", "question_arabic"),
           "B: BM25 + Roman query":         ("BM25", "question_roman"),
           "C: Skeleton + Roman query (ours)": ("Skeleton", "question_roman")}
for name, (retr, col) in SYSTEMS.items():
    run_config(name, retr, col, "zero-shot", seed=0)

A: BM25 + Arabic query|zero-shot|seed0: 150 answers (150 new) | total GPU seconds in file: 103
B: BM25 + Roman query|zero-shot|seed0: 150 answers (150 new) | total GPU seconds in file: 84
C: Skeleton + Roman query (ours)|zero-shot|seed0: 150 answers (150 new) | total GPU seconds in file: 103


In [40]:
for name, (retr, col) in SYSTEMS.items():
    for seed in SEEDS:
        run_config(name, retr, col, "few-shot", seed=seed)

C: Skeleton + Roman query (ours)|few-shot|seed0: 150 answers (150 new) | total GPU seconds in file: 129
C: Skeleton + Roman query (ours)|few-shot|seed1: 150 answers (150 new) | total GPU seconds in file: 123
C: Skeleton + Roman query (ours)|few-shot|seed2: 150 answers (150 new) | total GPU seconds in file: 119
A: BM25 + Arabic query|few-shot|seed0: 150 answers (0 new) | total GPU seconds in file: 139
A: BM25 + Arabic query|few-shot|seed1: 150 answers (0 new) | total GPU seconds in file: 131
A: BM25 + Arabic query|few-shot|seed2: 150 answers (0 new) | total GPU seconds in file: 132
B: BM25 + Roman query|few-shot|seed0: 150 answers (0 new) | total GPU seconds in file: 113
B: BM25 + Roman query|few-shot|seed1: 150 answers (0 new) | total GPU seconds in file: 105
B: BM25 + Roman query|few-shot|seed2: 150 answers (0 new) | total GPU seconds in file: 106
C: Skeleton + Roman query (ours)|few-shot|seed0: 150 answers (0 new) | total GPU seconds in file: 129
C: Skeleton + Roman query (ours)|few-

In [41]:
retr, col = SYSTEMS["C: Skeleton + Roman query (ours)"]
for seed in SEEDS:
    run_config("D: C + faithfulness check (ours, full)", retr, col, "few-shot", seed=seed, verify_answers=True)

D: C + faithfulness check (ours, full)|few-shot|seed0: 150 answers (150 new) | total GPU seconds in file: 219
D: C + faithfulness check (ours, full)|few-shot|seed1: 150 answers (150 new) | total GPU seconds in file: 159
D: C + faithfulness check (ours, full)|few-shot|seed2: 150 answers (150 new) | total GPU seconds in file: 155


### Step 3: results table (mean ± std over 3 seeds for few-shot)

In [42]:
def table():
    rows = []
    for cfg in sorted({k.split("|")[0] for k in RUNS}):
        for mode in ["zero-shot", "few-shot"]:
            ms = [summarize(per_question(RUNS[k])) for k in RUNS if k.startswith(f"{cfg}|{mode}|")]
            if not ms: continue
            row = {"system": cfg, "mode": mode, "n_runs": len(ms)}
            for m in ms[0]:
                v = [x[m] for x in ms]; row[m] = f"{np.nanmean(v):.3f} ± {np.nanstd(v):.3f}" if len(v) > 1 else f"{v[0]:.3f}"
            rows.append(row)
    return pd.DataFrame(rows)
gen_table = table(); gen_table

,system,mode,n_runs,Acc_answerable,F1_answerable,Abstain_acc_unans,Halluc_unans,Unsupported_rate,Abstain_rate_answerable
0,A: BM25 + Arabic query,zero-shot,1,0.504,0.427,0.241,0.759,0.106,0.017
1,A: BM25 + Arabic query,few-shot,3,0.562 ± 0.024,0.497 ± 0.020,0.322 ± 0.033,0.678 ± 0.033,0.124 ± 0.022,0.033 ± 0.000
2,B: BM25 + Roman query,zero-shot,1,0.066,0.059,0.586,0.414,0.036,0.645
3,B: BM25 + Roman query,few-shot,3,0.077 ± 0.004,0.065 ± 0.001,0.828 ± 0.049,0.172 ± 0.049,0.127 ± 0.047,0.780 ± 0.008
4,C: Skeleton + Roman query (ours),zero-shot,1,0.388,0.325,0.448,0.552,0.125,0.140
5,C: Skeleton + Roman query (ours),few-shot,3,0.452 ± 0.004,0.411 ± 0.002,0.575 ± 0.071,0.425 ± 0.071,0.095 ± 0.015,0.176 ± 0.014
6,"D: C + faithfulness check (ours, full)",few-shot,3,0.085 ± 0.004,0.073 ± 0.004,0.943 ± 0.016,0.057 ± 0.016,0.000 ± 0.000,0.906 ± 0.004


### Step 4: significance (paired bootstrap), ours vs best baseline, few-shot seed 0

In [43]:
def flags(cfg, mode="few-shot", seed=0):
    return per_question(RUNS[f"{cfg}|{mode}|seed{seed}"]).set_index("id").correct.astype(int)
base_names = [k.split("|")[0] for k in RUNS if k.startswith(("A:", "B:")) and "|few-shot|seed0" in k]
ours_name = "C: Skeleton + Roman query (ours)"
ours = flags(ours_name)
for b in base_names:
    bf = flags(b).reindex(ours.index)
    d, ci, p = paired_bootstrap(ours.values, bf.values)
    print(f"{ours_name} vs {b}: accuracy-of-behaviour diff = {d:+.3f}, 95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}], p = {p:.4f}")
print("(correct = answer matches gold for answerable questions, or abstains for unanswerable ones)")

C: Skeleton + Roman query (ours) vs A: BM25 + Arabic query: accuracy-of-behaviour diff = -0.033, 95% CI [-0.127, +0.060], p = 0.5440
C: Skeleton + Roman query (ours) vs B: BM25 + Roman query: accuracy-of-behaviour diff = +0.280, 95% CI [+0.187, +0.373], p = 0.0000
(correct = answer matches gold for answerable questions, or abstains for unanswerable ones)


### Step 5: fairness (script and topic) and cost

In [44]:
rows = []
for k in RUNS:
    if "|few-shot|seed0" not in k: continue
    pq = per_question(RUNS[k])
    for tp, g in pq.groupby("topic"):
        rows.append({"system": k.split("|")[0], "topic": tp, "n": len(g), "correct": round(g.correct.mean(), 3)})
topic_df = pd.DataFrame(rows).pivot(index="topic", columns="system", values="correct"); topic_df

system,A: BM25 + Arabic query,B: BM25 + Roman query,C: Skeleton + Roman query (ours),"D: C + faithfulness check (ours, full)"
topic,,,,
geography,0.432,0.378,0.486,0.378
health,0.333,0.333,0.500,0.333
history_culture,0.565,0.141,0.457,0.196
science,0.600,0.200,0.733,0.200


In [45]:
cost = []
for k, recs in RUNS.items():
    cost.append({"run": k, "questions": len(recs), "gpu_seconds": round(sum(r["secs"] for r in recs)),
                 "sec_per_question": round(np.mean([r["secs"] for r in recs]), 2),
                 "mean_prompt_tokens": round(np.mean([r["prompt_tokens"] for r in recs])), "api_cost_usd": 0.0})
cost_df = pd.DataFrame(cost); cost_df

,run,questions,gpu_seconds,sec_per_question,mean_prompt_tokens,api_cost_usd
0,A: BM25 + Arabic query|zero-shot|seed0,150,103,0.68,766,0.0
1,B: BM25 + Roman query|zero-shot|seed0,150,84,0.56,877,0.0
2,C: Skeleton + Roman query (ours)|zero-shot|seed0,150,103,0.69,735,0.0
3,A: BM25 + Arabic query|few-shot|seed0,150,139,0.93,1548,0.0
4,A: BM25 + Arabic query|few-shot|seed1,150,131,0.87,1473,0.0
5,A: BM25 + Arabic query|few-shot|seed2,150,132,0.88,1404,0.0
6,B: BM25 + Roman query|few-shot|seed0,150,113,0.75,1659,0.0
7,B: BM25 + Roman query|few-shot|seed1,150,105,0.70,1584,0.0
8,B: BM25 + Roman query|few-shot|seed2,150,106,0.70,1515,0.0
9,C: Skeleton + Roman query (ours)|few-shot|seed0,150,129,0.86,1517,0.0


### Step 6: errors to classify by hand (you need 50+ for the error taxonomy)

In [46]:
bad = []
for k in RUNS:
    if k.startswith("C:") and "|few-shot|seed0" in k:
        pq = per_question(RUNS[k]).set_index("id")
        for r in RUNS[k]:
            if not pq.loc[r["id"], "correct"]:
                t = test[test.id == r["id"]].iloc[0]
                bad.append({"id": r["id"], "question": t.question_arabic, "gold": t.gold_answer, "answerable": t.answerable,
                            "system_answer": r["answer"], "gold_paragraph_retrieved": t.para_id in r["retrieved"],
                            "retrieved_context": " ||| ".join(pid2text[p][:300] for p in r["retrieved"]),
                            "error_category": "", "english_translation": ""})
errs = pd.DataFrame(bad); errs.to_csv("errors_to_label.csv", index=False, encoding="utf-8-sig")
print(len(errs), "errors saved. Fill error_category (e.g. retrieval miss, wrong span, ignored context, number/name changed, answered unanswerable, abstained wrongly) and a translation for 50+ of them.")

76 errors saved. Fill error_category (e.g. retrieval miss, wrong span, ignored context, number/name changed, answered unanswerable, abstained wrongly) and a translation for 50+ of them.


### Step 7: human labels for hallucination (you and a second annotator)
This creates two sheets with the SAME 100 items in different orders: `labels_annotator1.csv` (you) and `labels_annotator2.csv` (the second person, who must not see your labels).
Labels: FAITHFUL (every claim is supported by the retrieved context), HALLUCINATED (something is not supported or contradicts the context), judged ONLY from the context. Abstained answers are excluded.

In [47]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
RULES = [('Rule 1', 'Judge ONLY against the retrieved context. Do not use outside knowledge, even if the answer is true in the real world.'), ('Rule 2', 'FAITHFUL = every claim in the system answer is supported by the retrieved context.'), ('Rule 3', 'HALLUCINATED = at least one claim is not in the context or contradicts it. If part is supported and part is not, choose HALLUCINATED.'), ('Rule 4', 'Numbers, names, dates and places must match the context exactly. Ignore grammar/spelling mistakes; judge meaning only.'), ('Rule 5', 'If unsure, choose HALLUCINATED and write the reason in notes. ABSTAINED is only for answers saying the information is not available.'), ('Do NOT', "look at the other annotator's labels. Work alone.")]
def make_xlsx(df, path):
    wb = Workbook(); ws = wb.active; ws.title = "Labels"
    cols = ["item_id", "question", "retrieved_context", "system_answer", "label", "notes"]
    ws.append(cols)
    for _, r in df[cols].iterrows(): ws.append([str(r[c]) if str(r[c]) != "nan" else "" for c in cols])
    for c in ws[1]: c.font = Font(bold=True); c.fill = PatternFill("solid", fgColor="DDDDDD")
    for k, v in {"A": 9, "B": 45, "C": 90, "D": 30, "E": 16, "F": 30}.items(): ws.column_dimensions[k].width = v
    for row in ws.iter_rows(min_row=2):
        for c in row:
            rtl = c.column in (2, 3, 4)
            c.alignment = Alignment(wrap_text=True, vertical="top", readingOrder=2 if rtl else 0, horizontal="right" if rtl else "left")
    ws.freeze_panes = "B2"
    dv = DataValidation(type="list", formula1='"FAITHFUL,HALLUCINATED,ABSTAINED"', allow_blank=True)
    ws.add_data_validation(dv); dv.add(f"E2:E{ws.max_row}")
    r = wb.create_sheet("Rules"); r.column_dimensions["A"].width = 10; r.column_dimensions["B"].width = 130
    for a, b in RULES: r.append([a, b])
    wb.save(path)

ref_key = "C: Skeleton + Roman query (ours)|few-shot|seed0"
cand = []
for r in RUNS[ref_key]:
    if r["abstained"]: continue
    t = test[test.id == r["id"]].iloc[0]
    cand.append({"item_id": r["id"], "question": t.question_arabic,
                 "retrieved_context": "\n\n".join(f"[{i+1}] {pid2text[p]}" for i, p in enumerate(r["retrieved"])),
                 "system_answer": r["answer"], "label": "", "notes": ""})
cand = pd.DataFrame(cand)
n_items = min(100, len(cand))
items = cand.sample(n=n_items, random_state=0)
items.sample(frac=1, random_state=1).pipe(lambda d: make_xlsx(d, "labels_annotator1.xlsx"))
items.sample(frac=1, random_state=2).pipe(lambda d: make_xlsx(d, "labels_annotator2.xlsx"))
print(f"{n_items} items written (non-abstained answers available: {len(cand)}).")
print("Send labels_annotator2.xlsx to the second annotator together with the guidelines. Download both files:")
files.download("labels_annotator1.xlsx"); files.download("labels_annotator2.xlsx")

100 items written (non-abstained answers available: 112).
Send labels_annotator2.xlsx to the second annotator together with the guidelines. Download both files:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Step 8: agreement (Cohen's kappa) and validation of the automatic proxy
Run this after both annotators have filled `label` and you uploaded the two files with the same names.

In [48]:
import io
from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score
def read_labels(stem):
    """Reads <stem>.xlsx (preferred) or <stem>.csv; tolerates Excel re-saves (tab separated / ANSI)."""
    for suffix in ("", "_completed"):
        if os.path.exists(stem + suffix + ".xlsx"):
            return pd.read_excel(stem + suffix + ".xlsx", sheet_name="Labels").fillna("").astype(str)
    if os.path.exists(stem + ".csv"):
        raw = open(stem + ".csv", "rb").read()
        for enc in ["utf-8-sig", "utf-16", "cp1252"]:
            try: txt = raw.decode(enc); break
            except UnicodeDecodeError: continue
        sep = "\t" if txt.count("\t") > txt.count(",") else ","
        return pd.read_csv(io.StringIO(txt), sep=sep).fillna("").astype(str)
    return None
for stem in ["labels_annotator1", "labels_annotator2"]:
    if read_labels(stem) is None: print("Upload the filled", stem, ".xlsx (or ", stem, "_completed.xlsx)"); files.upload()
a1, a2 = read_labels("labels_annotator1"), read_labels("labels_annotator2")
for a in (a1, a2): a["label"] = a.label.str.strip().str.upper()
print("labels filled: annotator1", (a1.label != "").sum(), "| annotator2", (a2.label != "").sum())
m = a1.merge(a2[["item_id", "label"]], on="item_id", suffixes=("_1", "_2"))
ok = m.label_1.isin(["FAITHFUL", "HALLUCINATED"]) & m.label_2.isin(["FAITHFUL", "HALLUCINATED"])
m = m[ok]
print("items labelled by both:", len(m))
if len(m) >= 50:
    print("Cohen's kappa:", round(cohen_kappa_score(m.label_1, m.label_2), 3),
          "| raw agreement:", round((m.label_1 == m.label_2).mean(), 3))
    sup = {r["id"]: support(r["answer"], [pid2text[p] for p in r["retrieved"]]) for r in RUNS[ref_key]}
    m["proxy"] = m.item_id.map(lambda i: "HALLUCINATED" if sup[i] < 0.5 else "FAITHFUL")
    for who in ["label_1", "label_2"]:
        print(f"proxy vs {who}: accuracy {accuracy_score(m[who], m.proxy):.3f}, macro-F1 {f1_score(m[who], m.proxy, average='macro'):.3f}")
    print("human HALLUCINATED rate (annotator 1):", round((m.label_1 == "HALLUCINATED").mean(), 3))
else:
    print("Need at least 50 items labelled by both annotators.")

labels filled: annotator1 0 | annotator2 0
items labelled by both: 0
Need at least 50 items labelled by both annotators.


In [49]:
gen_table.to_csv("generation_results.csv", index=False); topic_df.to_csv("generation_by_topic.csv")
cost_df.to_csv("generation_cost.csv", index=False)
import shutil; shutil.make_archive("gen_outputs", "zip", "gen_outputs")
for f in ["generation_results.csv", "generation_by_topic.csv", "generation_cost.csv", "errors_to_label.csv", "gen_outputs.zip"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Part 3C: Hallucination judges at three levels (trained / evaluated on the human labels)
Teacher's baseline levels applied to the faithfulness judge: (a) classical TF-IDF character n-grams + logistic regression, (b) fine-tuned multilingual encoder XLM-R, (c) prompted LLM (zero-shot). The automatic proxy from Part 3B is reported as well.
Only ~100 labelled items exist, so (a) and (b) use stratified 5-fold cross-validation repeated over 3 seeds; this is a limitation to state in the paper.
Labels come from `labels_annotator1` (the reference annotator). Run after Step 8 of Part 3B.

In [56]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
J = read_labels("labels_annotator1")
J["label"] = J.label.str.strip().str.upper()
J = J[J.label.isin(["FAITHFUL", "HALLUCINATED"])].reset_index(drop=True)
J["y"] = (J.label == "HALLUCINATED").astype(int)
J["text"] = "Q: " + J.question + " A: " + J.system_answer + " C: " + J.retrieved_context
print("judge dataset:", len(J), "| hallucinated share:", round(J.y.mean(), 3))
judge_rows = []

judge dataset: 100 | hallucinated share: 0.49


### Judge (a): TF-IDF character n-grams + logistic regression

In [57]:
t0 = time.time(); scores = []
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed); pred = np.zeros(len(J), dtype=int)
    for tr, te in skf.split(J.text, J.y):
        vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
        Xtr = vec.fit_transform(J.text[tr]); clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=seed).fit(Xtr, J.y[tr])
        pred[te] = clf.predict(vec.transform(J.text[te]))
    scores.append(f1_score(J.y, pred, average="macro"))
judge_rows.append({"judge": "TF-IDF char n-gram + LR (5-fold CV)", "macro_F1": f"{np.mean(scores):.3f} ± {np.std(scores):.3f}",
                   "seconds": round(time.time() - t0, 1), "gpu": False})
judge_rows[-1]

{'judge': 'TF-IDF char n-gram + LR (5-fold CV)',
 'macro_F1': '0.516 ± 0.041',
 'seconds': 15.9,
 'gpu': False}

### Judge (b): fine-tuned XLM-R base (5-fold CV x 3 seeds, GPU)

In [58]:
import torch
from transformers import AutoTokenizer as _AT, AutoModelForSequenceClassification
XLMR = "xlm-roberta-base"
xtok = _AT.from_pretrained(XLMR)
def enc_batch(df, max_len=384):
    return xtok(list(("Q: " + df.question + " A: " + df.system_answer).values), list(df.retrieved_context.values),
                truncation="only_second", max_length=max_len, padding=True, return_tensors="pt")
def train_eval(tr_df, te_df, seed, epochs=6, lr=2e-5, bs=8):
    torch.manual_seed(seed); np.random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(XLMR, num_labels=2).cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    ytr = torch.tensor(tr_df.y.values)
    for ep in range(epochs):
        model.train(); idx = np.random.permutation(len(tr_df))
        for i in range(0, len(idx), bs):
            b = idx[i:i + bs]; e = {k: v.cuda() for k, v in enc_batch(tr_df.iloc[b]).items()}
            loss = model(**e, labels=ytr[b].cuda()).loss; loss.backward(); opt.step(); opt.zero_grad()
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(te_df), 16):
            e = {k: v.cuda() for k, v in enc_batch(te_df.iloc[i:i + 16]).items()}
            out += model(**e).logits.argmax(-1).cpu().tolist()
    del model; torch.cuda.empty_cache()
    return np.array(out)

t0 = time.time(); scores = []
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed); pred = np.zeros(len(J), dtype=int)
    for tr, te in skf.split(J.text, J.y):
        pred[te] = train_eval(J.iloc[tr].reset_index(drop=True), J.iloc[te].reset_index(drop=True), seed)
    scores.append(f1_score(J.y, pred, average="macro")); print("seed", seed, "macro-F1", round(scores[-1], 3))
judge_rows.append({"judge": "XLM-R base fine-tuned (5-fold CV)", "macro_F1": f"{np.mean(scores):.3f} ± {np.std(scores):.3f}",
                   "seconds": round(time.time() - t0, 1), "gpu": True})
judge_rows[-1]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

... (30 outputs left out)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


seed 2 macro-F1 0.355


{'judge': 'XLM-R base fine-tuned (5-fold CV)',
 'macro_F1': '0.406 ± 0.094',
 'seconds': 637.8,
 'gpu': True}

### Judge (c): prompted LLM (zero-shot, same Qwen model as the generator)

In [59]:
def llm_judge(question, context, answer):
    msg = [{"role": "system", "content": VERIFY_SYSTEM},
           {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}\nProposed answer: {answer}\n\n"
                                       "Is the proposed answer fully supported by the context? Reply YES or NO."}]
    out, _ = generate(msg, max_new_tokens=3)
    return 0 if out.strip().upper().startswith("YES") else 1     # 1 = hallucinated
t0 = time.time()
pred = np.array([llm_judge(r.question, r.retrieved_context, r.system_answer) for r in J.itertuples()])
judge_rows.append({"judge": "Qwen2.5-3B zero-shot judge", "macro_F1": f"{f1_score(J.y, pred, average='macro'):.3f}",
                   "seconds": round(time.time() - t0, 1), "gpu": True})
judge_rows[-1]

{'judge': 'Qwen2.5-3B zero-shot judge',
 'macro_F1': '0.509',
 'seconds': 32.8,
 'gpu': True}

### Judge (d): the automatic proxy from Part 3B (token support < 0.5 means unsupported)

In [60]:
def proxy_pred(r):
    ctx = re.split(r"\[\d\]", r.retrieved_context)
    return int(support(r.system_answer, ctx) < 0.5)
pred = np.array([proxy_pred(r) for r in J.itertuples()])
judge_rows.append({"judge": "Token-support proxy (no training)", "macro_F1": f"{f1_score(J.y, pred, average='macro'):.3f}",
                   "seconds": 0.0, "gpu": False})
judge_df = pd.DataFrame(judge_rows); judge_df.to_csv("judge_results.csv", index=False)
files.download("judge_results.csv"); judge_df

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,judge,macro_F1,seconds,gpu
0,TF-IDF char n-gram + LR (5-fold CV),0.516 ± 0.041,15.9,False
1,XLM-R base fine-tuned (5-fold CV),0.406 ± 0.094,637.8,True
2,Qwen2.5-3B zero-shot judge,0.509,32.8,True
3,Token-support proxy (no training),0.466,0.0,False
